# 02 — Monthly Forecast Band Plot

Produces a **P10–P90 shaded band + mean line** chart for monthly generation or revenue across the forecast horizon — the style shown in the performance dashboard screenshot.

Reads from `generation.duckdb` / `revenue.duckdb` (`monthly` table, `eligible_for_*_dist = TRUE`).

**Steps:**
1. Config
2. Data loading (GCS or local)
3. Monthly aggregation (P10/P25/P50/P75/P90/mean per calendar month)
4. Band plot — matches screenshot style
5. Scaffold: per-segment breakdown (historical vs simulated)

> Monthly KDE / distribution analysis will be added in a future notebook.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1: Configuration
# ═══════════════════════════════════════════════════════════════════

# Site / market to analyse
SITE      = "ash_creek_solar"
KIND      = "hub"   # 'hub' or 'node'
MARKET    = "da"    # 'da' or 'rt'

# Which variable to plot
VARIABLE  = "generation"   # 'generation' or 'revenue'

# Data source
SOURCE    = "gcs"   # 'gcs' or 'local'

# Paths
import os
from pathlib import Path
REPO_ROOT      = Path(os.environ.get("MODEL_GPR_ROOT",
                       "/Users/divy/code/work/infrasure_git_codes/model-gpr")).resolve()
LOCAL_AGG_DIR  = REPO_ROOT / "local_data" / "aggregated_data"
GCS_BUCKET     = "infrasure-model-gpr-data"
GCS_PROJECT    = "infrasure-model-gpr"
GCS_AGG_PREFIX = "aggregated_data"

print(f"Site: {SITE}  |  Kind: {KIND}  |  Market: {MARKET}")
print(f"Variable: {VARIABLE}  |  Source: {SOURCE}")

Site: ash_creek_solar  |  Kind: hub  |  Market: da
Variable: generation  |  Source: gcs


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2: Data loading
# ═══════════════════════════════════════════════════════════════════

import warnings, tempfile, shutil
import numpy as np
import pandas as pd
import duckdb

warnings.filterwarnings("ignore", message=".*end user credentials.*")
warnings.filterwarnings("ignore", message=".*quota project.*")

def _get_duckdb_as_temp(db_name: str) -> str:
    tmp = tempfile.mktemp(suffix=".duckdb")
    if SOURCE == "local":
        src = LOCAL_AGG_DIR / SITE / db_name
        if not src.exists():
            raise FileNotFoundError(f"Not found locally: {src}")
        shutil.copy2(str(src), tmp)
    else:
        from google.cloud import storage
        client    = storage.Client(project=GCS_PROJECT)
        bucket    = client.bucket(GCS_BUCKET)
        blob_path = f"{GCS_AGG_PREFIX}/{SITE}/{db_name}"
        blob = bucket.blob(blob_path)
        if not blob.exists():
            raise FileNotFoundError(
                f"Not found in GCS: gs://{GCS_BUCKET}/{blob_path}\n"
                "Run forecast-aggregation first."
            )
        blob.download_to_filename(tmp)
    return tmp

if VARIABLE == "generation":
    _db_name    = "generation.duckdb"
    _val_col    = "monthly_generation_mwh"
    _elig_col   = "eligible_for_gen_dist"
    _unit       = "MWh"
    _scale      = 1.0
    _y_label    = "Generation (MWh)"
    _color_band = "rgba(0,204,150,0.25)"
    _color_line = "#00cc96"
else:
    _db_name    = "revenue.duckdb"
    _val_col    = "monthly_revenue_usd"
    _elig_col   = "eligible_for_rev_dist"
    _unit       = "USD"
    _scale      = 1e-3  # → kUSD
    _y_label    = "Revenue (k USD)"
    _color_band = "rgba(99,110,250,0.22)"
    _color_line = "#636efa"

_tmp = _get_duckdb_as_temp(_db_name)
conn = duckdb.connect(_tmp, read_only=True)

monthly = conn.execute(f"""
    SELECT path_id, segment, year_month, {_val_col}
    FROM monthly
    WHERE kind=? AND market=?
      AND {_elig_col} = TRUE
      AND {_val_col} IS NOT NULL
    ORDER BY year_month, path_id
""", [KIND, MARKET]).df()

conn.close()
import os as _os
try: _os.unlink(_tmp)
except: pass

monthly["value"] = monthly[_val_col] * _scale
print(f"Loaded {len(monthly)} eligible monthly rows  "
      f"({monthly['path_id'].nunique()} paths, "
      f"{monthly['year_month'].nunique()} months)")

Loaded 1759 eligible monthly rows  (147 paths, 12 months)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3: Monthly aggregation — percentiles + mean
# ═══════════════════════════════════════════════════════════════════

def _month_agg(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby("year_month")["value"]
    agg = pd.DataFrame({
        "p10":  grp.quantile(0.10),
        "p25":  grp.quantile(0.25),
        "p50":  grp.quantile(0.50),
        "p75":  grp.quantile(0.75),
        "p90":  grp.quantile(0.90),
        "mean": grp.mean(),
        "n":    grp.count(),
    }).reset_index()
    # Parse year_month to a date for clean x-axis
    agg["date"] = pd.to_datetime(agg["year_month"] + "-01")
    return agg.sort_values("date")

# All eligible paths
agg_all  = _month_agg(monthly)
# Simulated only
agg_sim  = _month_agg(monthly[monthly["segment"] == "simulated"])
# Historical only
agg_hist = _month_agg(monthly[monthly["segment"] == "historical"])

print(f"Months in horizon: {agg_all['year_month'].tolist()}")
print()
print(agg_all[["year_month","p10","p50","mean","p90","n"]].to_string(index=False))

Months in horizon: ['2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10', '2026-11', '2026-12', '2027-01']

year_month          p10          p50         mean           p90   n
   2026-02 32290.692462 38716.062336 39234.137022  45213.654278 147
   2026-03 39921.987900 53178.194995 51560.169649  58549.246095 147
   2026-04 49595.762573 56334.261092 56064.878107  61899.089399 147
   2026-05 58036.669034 75649.426532 72451.479424  79430.521452 147
   2026-06 82747.179564 94487.695727 94219.133275 102126.862966 147
   2026-07 92802.669387 99517.837242 99682.160285 107675.583559 147
   2026-08 80214.494212 93516.806454 92336.578182 100717.315787 146
   2026-09 67344.347604 77984.164067 76792.573996  85085.316456 146
   2026-10 48979.713385 56941.962278 55712.209742  62874.631889 146
   2026-11 32520.077963 38429.512497 37962.544787  43533.615222 146
   2026-12 26391.085755 34945.281938 34318.754547  41690.885781 146
   2027-01 37010.951274 42179.5

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4: Band plot — P10–P90 range + mean line
#         + Simulated mean vs Historical mean overlay for comparison
#         Matches the dashboard screenshot style
# ═══════════════════════════════════════════════════════════════════

import plotly.graph_objects as go

def _band_plot(
    agg: pd.DataFrame,
    title: str,
    band_color: str = _color_band,
    line_color: str = _color_line,
    y_label: str = _y_label,
    band_label: str = "P10–P90 Range",
    mean_label: str = "Mean",
    p50_label: str = "Median (P50)",
) -> go.Figure:
    x = agg["date"].dt.strftime("%b %Y")
    fig = go.Figure()

    # Shaded P10–P90 band
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["date"], agg["date"][::-1]]).dt.strftime("%b %Y"),
        y=pd.concat([agg["p90"], agg["p10"][::-1]]),
        fill="toself",
        fillcolor=band_color,
        line=dict(color="rgba(0,0,0,0)"),
        hoverinfo="skip",
        name=band_label,
        showlegend=True,
    ))

    # P25–P75 inner band (slightly darker)
    inner_color = band_color.replace("0.22", "0.40").replace("0.25", "0.45")
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["date"], agg["date"][::-1]]).dt.strftime("%b %Y"),
        y=pd.concat([agg["p75"], agg["p25"][::-1]]),
        fill="toself",
        fillcolor=inner_color,
        line=dict(color="rgba(0,0,0,0)"),
        hoverinfo="skip",
        name="P25–P75 Range",
        showlegend=True,
    ))

    # Mean line
    fig.add_trace(go.Scatter(
        x=x, y=agg["mean"],
        mode="lines+markers",
        name=mean_label,
        line=dict(color=line_color, width=2.5),
        marker=dict(size=5),
    ))

    # P50 line (dashed)
    fig.add_trace(go.Scatter(
        x=x, y=agg["p50"],
        mode="lines",
        name=p50_label,
        line=dict(color=line_color, width=1.5, dash="dot"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title=y_label,
        template="plotly_white",
        legend=dict(
            orientation="h", y=-0.2,
            xanchor="left", x=0,
        ),
        hovermode="x unified",
        width=860, height=480,
    )
    return fig

# Band = simulated paths only (the forecast distribution)
# Historical mean = separate overlay so you can compare
fig_all = _band_plot(
    agg_sim,
    title=f"Monthly {VARIABLE.capitalize()} Forecast — {SITE}  {KIND}/{MARKET}<br>"
          f"<sup>Band = simulated paths | Blue = simulated mean | Green dashed = historical mean</sup>",
    band_label="P10–P90 (simulated)",
    mean_label="Simulated mean",
    p50_label="Simulated P50",
)

# Historical mean overlay
if not agg_hist.empty:
    fig_all.add_trace(go.Scatter(
        x=agg_hist["date"].dt.strftime("%b %Y"),
        y=agg_hist["mean"],
        mode="lines+markers",
        name="Historical mean",
        line=dict(color="#00cc96", width=2.5, dash="dash"),
        marker=dict(size=5, symbol="square"),
    ))

fig_all.show()

# Print mean comparison
if not agg_sim.empty and not agg_hist.empty:
    merged = agg_sim[["year_month","mean"]].rename(columns={"mean":"sim_mean"}).merge(
        agg_hist[["year_month","mean"]].rename(columns={"mean":"hist_mean"}),
        on="year_month", how="inner"
    )
    merged["diff"] = merged["sim_mean"] - merged["hist_mean"]
    merged["diff_pct"] = (merged["diff"] / merged["hist_mean"] * 100).round(1)
    print(f"\nSimulated vs Historical mean comparison  ({_unit}):")
    print(merged[["year_month","sim_mean","hist_mean","diff","diff_pct"]].to_string(index=False))


Simulated vs Historical mean comparison  (MWh):
year_month     sim_mean     hist_mean         diff  diff_pct
   2026-02 39511.162888  38644.720287   866.442601       2.2
   2026-03 51680.840642  51303.422856   377.417785       0.7
   2026-04 55639.592952  56969.740139 -1330.147186      -2.3
   2026-05 72481.608393  72387.375235    94.233158       0.1
   2026-06 95044.647520  92462.719987  2581.927533       2.8
   2026-07 99505.761503 100057.476844  -551.715341      -0.6
   2026-08 91832.119234  93433.228070 -1601.108836      -1.7
   2026-09 77098.388086  76127.760755   970.627331       1.3
   2026-10 56798.669308  53350.341122  3448.328186       6.5
   2026-11 38334.120112  37154.772342  1179.347770       3.2
   2026-12 34452.744500  34027.472042   425.272458       1.2
   2027-01 42725.142493  40436.075106  2289.067387       5.7


In [5]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5: Simulated-only vs Historical-only band comparison
# ═══════════════════════════════════════════════════════════════════

from plotly.subplots import make_subplots

fig_seg = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Simulated paths only", "Historical paths only"],
    horizontal_spacing=0.08,
)

def _add_band_traces(fig, agg, band_color, line_color, col):
    if agg.empty:
        return
    x = agg["date"].dt.strftime("%b %Y")
    # P10–P90 band
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["date"], agg["date"][::-1]]).dt.strftime("%b %Y"),
        y=pd.concat([agg["p90"], agg["p10"][::-1]]),
        fill="toself", fillcolor=band_color,
        line=dict(color="rgba(0,0,0,0)"),
        name="P10–P90", showlegend=col==1
    ), row=1, col=col)
    # Mean
    fig.add_trace(go.Scatter(
        x=x, y=agg["mean"], mode="lines+markers",
        name="Mean", line=dict(color=line_color, width=2.5), marker=dict(size=5),
        showlegend=col==1
    ), row=1, col=col)

_sim_band   = _color_band
_hist_band  = "rgba(0,204,150,0.25)"
_hist_line  = "#00cc96"

_add_band_traces(fig_seg, agg_sim,  _sim_band, _color_line, col=1)
_add_band_traces(fig_seg, agg_hist, _hist_band, _hist_line,  col=2)

fig_seg.update_layout(
    title=f"Monthly {VARIABLE.capitalize()} — Simulated vs Historical  |  {SITE}  {KIND}/{MARKET}",
    yaxis_title=_y_label, yaxis2_title=_y_label,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.2),
    width=1050, height=460,
)
fig_seg.show()

print("\nSimulated paths  — monthly P50 range:")
if not agg_sim.empty:
    print(f"  {agg_sim['year_month'].iloc[0]}: {agg_sim['p50'].iloc[0]:,.0f} {_unit}")
    print(f"  {agg_sim['year_month'].iloc[-1]}: {agg_sim['p50'].iloc[-1]:,.0f} {_unit}")
print("\nHistorical paths — monthly P50 range:")
if not agg_hist.empty:
    print(f"  {agg_hist['year_month'].iloc[0]}: {agg_hist['p50'].iloc[0]:,.0f} {_unit}")
    print(f"  {agg_hist['year_month'].iloc[-1]}: {agg_hist['p50'].iloc[-1]:,.0f} {_unit}")


Simulated paths  — monthly P50 range:
  2026-02: 38,682 MWh
  2027-01: 42,318 MWh

Historical paths — monthly P50 range:
  2026-02: 38,759 MWh
  2027-01: 40,951 MWh


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6: Recent years — individual monthly lines on the simulated band
#         Shows the last N historical years as separate lines so you
#         can see the trend and where recent years sit vs the distribution.
# ═══════════════════════════════════════════════════════════════════

N_RECENT_YEARS = 3   # ← change to show more/fewer recent years

# Get distinct historical source_years, pick the N most recent
hist_monthly = monthly[monthly["segment"] == "historical"].copy()

# For historical paths, path_id = source year
hist_monthly["source_year"] = hist_monthly["path_id"]
recent_years = sorted(hist_monthly["source_year"].unique())[-N_RECENT_YEARS:]

print(f"Recent years shown: {recent_years}")

# Palette for recent years (darkest = most recent)
palette = ["#fdae6b", "#e6550d", "#a63603"]   # light → dark orange
if len(recent_years) > len(palette):
    import colorsys
    palette = [f"hsl({int(200-i*30)},70%,50%)" for i in range(len(recent_years))]

fig_trend = _band_plot(
    agg_sim,
    title=f"Monthly {VARIABLE.capitalize()} — Simulated band + Recent {N_RECENT_YEARS} historical years<br>"
          f"<sup>{SITE}  {KIND}/{MARKET}  |  Orange lines = recent years (darker = more recent)</sup>",
)
fig_trend.update_traces(selector=dict(name="Mean"), line_color="#636efa")

# Add each recent year as a line
for yr, color in zip(recent_years, palette):
    yr_data = hist_monthly[hist_monthly["source_year"] == yr].copy()
    yr_data["date"] = pd.to_datetime(yr_data["year_month"] + "-01")
    yr_data = yr_data.sort_values("date")
    fig_trend.add_trace(go.Scatter(
        x=yr_data["date"].dt.strftime("%b %Y"),
        y=yr_data["value"],
        mode="lines+markers",
        name=f"Year {yr}",
        line=dict(color=color, width=2),
        marker=dict(size=6, symbol="circle"),
    ))

fig_trend.update_layout(width=900, height=500)
fig_trend.show()

# Print % vs simulated P50 for each recent year
print(f"\nRecent year vs simulated P50 (each month):")
p50_sim = agg_sim[["year_month","p50"]].copy()
for yr in recent_years:
    yr_data = hist_monthly[hist_monthly["source_year"] == yr][["year_month","value"]].rename(columns={"value": f"yr_{yr}"})
    p50_sim = p50_sim.merge(yr_data, on="year_month", how="left")
    p50_sim[f"vs_p50_{yr}"] = ((p50_sim[f"yr_{yr}"] - p50_sim["p50"]) / p50_sim["p50"] * 100).round(1)

cols = ["year_month"] + [f"vs_p50_{yr}" for yr in recent_years]
print(p50_sim[cols].to_string(index=False))
print(f"\n(positive = above simulated P50, negative = below)")

Recent years shown: [np.int32(2023), np.int32(2024), np.int32(2025)]



Recent year vs simulated P50 (each month):
year_month  vs_p50_2023  vs_p50_2024  vs_p50_2025
   2026-02         -0.1         12.7         -3.0
   2026-03         -7.7         -1.0         15.3
   2026-04         -7.7         -8.0         -0.5
   2026-05        -13.0        -10.8         -1.9
   2026-06         -2.7          4.6         -1.0
   2026-07          2.3          0.0         -5.2
   2026-08         11.1          9.4          NaN
   2026-09          2.7          3.6          NaN
   2026-10         -5.1         18.6          NaN
   2026-11         -6.6          9.8          NaN
   2026-12         16.3          3.0          NaN
   2027-01         -0.7          3.0          3.6

(positive = above simulated P50, negative = below)


---

## Scaffold: future monthly analysis

The following analyses will be added in an expanded version of this notebook:

- **Monthly KDE** — Scott KDE for each calendar month separately (Jan, Feb, … Dec), enabling seasonal distribution comparison
- **Month-by-month empirical CDF + KS-test** — same as annual analysis (Cell 5 of notebook 01) but per month
- **Seasonal decomposition** — compare summer vs winter months
- **Revenue seasonality** — monthly revenue mean ± 1 std, annotated with gen-weighted price by month
- **Correlation** — monthly generation vs monthly gen-weighted price scatter